# Approach A runbook: spaCy + classical ML

Run from the `resume_parser_spacy/` folder. Each cell calls a script, so the outputs below record training curves and metrics. The scripts are the source of truth.

| Step | Time (CPU) |
|---|---|
| Section classifier | ~1 min |
| NER (40 epochs max, early stopping) | ~18 min |
| Evaluations | ~3 min |

In [1]:
import platform, sys, spacy, sklearn, torch
print("Python", platform.python_version(), "| spaCy", spacy.__version__, "| scikit-learn", sklearn.__version__, "| torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

Python 3.10.12 | spaCy 3.8.16 | scikit-learn 1.7.2 | torch 2.5.0+cu124
GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU


## 1. Section classifier: TF-IDF + Logistic Regression

Trained on the Label Studio section annotations (9 resumes) plus every heading alias as a weak label. Prints 4-fold grouped cross-validation and held-out scores. Re-run it whenever the alias list in `preprocessing/sections.py` changes.

In [2]:
!python3 training/train_section_classifier.py

train blocks=369 test blocks=140 labels={'PERSONAL_INFORMATION': 32, 'SKILLS': 54, 'OTHER': 17, 'EDUCATION': 31, 'PROJECTS': 47, 'SUMMARY': 24, 'EXPERIENCE': 132, 'CERTIFICATIONS': 31, 'VOLUNTEERING': 1}
fold 1: {'accuracy': 0.5, 'macro_f1': 0.4884, 'weighted_f1': 0.4857}
fold 2: {'accuracy': 0.4182, 'macro_f1': 0.2708, 'weighted_f1': 0.4055}
fold 3: {'accuracy': 0.5252, 'macro_f1': 0.352, 'weighted_f1': 0.5292}
fold 4: {'accuracy': 0.6087, 'macro_f1': 0.499, 'weighted_f1': 0.6221}
                      precision    recall  f1-score   support

        ACHIEVEMENTS       0.00      0.00      0.00         2
      CERTIFICATIONS       0.50      0.67      0.57         6
           EDUCATION       0.50      0.64      0.56        11
          EXPERIENCE       0.39      0.57      0.47        47
     EXTRACURRICULAR       0.00      0.00      0.00         0
               OTHER       1.00      0.20      0.33         5
PERSONAL_INFORMATION       1.00      0.56      0.71         9
            PROJ

## 2. NER: spaCy CNN with static word vectors

Trains on the 248-resume train split. The best epoch is picked by validation exact-span F1, and training stops early after 6 epochs without improvement. The test split is not used. This overwrites `models/resume_ner_spacy_vectors/`, the model the parser loads. Needs `python -m spacy download en_core_web_md` once.

In [1]:
!python3 training/train_spacy_ner.py --output models/resume_ner_spacy_vectors --vectors en_core_web_md --epochs 40 --patience 6

Span alignment: train contracted=317, punctuation-trimmed=291, skipped=194; validation contracted=42, punctuation-trimmed=30, skipped=24
epoch 1/40 ner_loss=104883.281 dev_P=0.000 dev_R=0.000 dev_F1=0.000
epoch 2/40 ner_loss=22798.789 dev_P=0.144 dev_R=0.078 dev_F1=0.101
epoch 3/40 ner_loss=21693.188 dev_P=0.342 dev_R=0.161 dev_F1=0.219
epoch 4/40 ner_loss=20197.750 dev_P=0.326 dev_R=0.300 dev_F1=0.312
epoch 5/40 ner_loss=19635.611 dev_P=0.376 dev_R=0.286 dev_F1=0.325
epoch 6/40 ner_loss=18140.445 dev_P=0.428 dev_R=0.289 dev_F1=0.345
epoch 7/40 ner_loss=17613.074 dev_P=0.350 dev_R=0.451 dev_F1=0.394
epoch 8/40 ner_loss=16992.852 dev_P=0.477 dev_R=0.400 dev_F1=0.435
epoch 9/40 ner_loss=15726.545 dev_P=0.511 dev_R=0.402 dev_F1=0.450
epoch 10/40 ner_loss=15441.281 dev_P=0.469 dev_R=0.387 dev_F1=0.424
epoch 11/40 ner_loss=14432.769 dev_P=0.459 dev_R=0.494 dev_F1=0.476
epoch 12/40 ner_loss=14249.130 dev_P=0.478 dev_R=0.472 dev_F1=0.475
epoch 13/40 ner_loss=14191.376 dev_P=0.483 dev_R=0.452 

## Evaluate on the held-out test split

Run once, after training. **Entity level:** precision, recall and F1 per label, with exact and overlap matching. **Section level:** per-line accuracy, comparing rules only, classifier only and the hybrid. **Field and document level:** baseline vs full system. Each script also saves JSON to `evaluation/results/`.

In [1]:
!python3 evaluation/evaluate_ner.py

Approach A: spaCy CNN NER + TF-IDF/LogReg sections + char n-gram skill matching
Evaluated resumes: 31; mean NER time per resume: 73 ms
LABEL           EXACT P      R     F1   OVERLAP P      R     F1  SUPPORT
COMPANY           0.276  0.214  0.241       0.437  0.339  0.382      112
DATE              0.769  0.566  0.652       0.821  0.604  0.696      182
DEGREE            0.569  0.439  0.496       0.804  0.621  0.701       66
EMAIL             1.000  0.907  0.952       1.000  0.907  0.952       54
INSTITUTION       0.674  0.413  0.512       0.804  0.493  0.612       75
JOB_TITLE         0.527  0.458  0.490       0.754  0.656  0.702      192
LOCATION          0.603  0.429  0.502       0.706  0.503  0.588      177
PERSON            0.974  0.638  0.771       0.974  0.638  0.771       58
SKILL             0.433  0.420  0.426       0.515  0.500  0.507      576
----------------------------------------------------------------------
MICRO             0.540  0.455  0.494       0.648  0.546  0.593


In [2]:
!python3 evaluation/evaluate_sections.py

Approach A: spaCy CNN NER + TF-IDF/LogReg sections + char n-gram skill matching
Held-out resumes: 3  lines: 202
SYSTEM              ACCURACY  MACRO F1  WEIGHTED F1
rules only             0.579     0.416        0.530
classifier only        0.431     0.365        0.415


hybrid                 0.629     0.502        0.590

Hybrid per-section report:
                      precision    recall  f1-score   support

        ACHIEVEMENTS       0.00      0.00      0.00         2
      CERTIFICATIONS       1.00      0.67      0.80         6
           EDUCATION       0.44      1.00      0.61        11
          EXPERIENCE       0.56      0.97      0.71        63
               OTHER       0.20      0.09      0.12        11
PERSONAL_INFORMATION       1.00      0.50      0.67        14
            PROJECTS       1.00      0.82      0.90        34
              SKILLS       0.57      0.28      0.37        29
             SUMMARY       0.70      0.22      0.33        32

            accuracy                           0.63       202
           macro avg       0.61      0.50      0.50       202
        weighted avg       0.67      0.63      0.59       202

Saved /home/shaanal/Documents/EHS/Project_Resume_Parser/resume_parser_spacy/evaluation/results/sections.json


In [3]:
!python3 evaluation/evaluate_fields.py

Approach A: spaCy CNN NER + TF-IDF/LogReg sections + char n-gram skill matching
Resumes: 31
FIELD            baseline (rules + ontology)                   full system
full_name                              0.857                         0.893
email                                  1.000                         1.000
location                               0.567                         0.567
skills                                 0.376                         0.456
degree                                 0.603                         0.756
institution                            0.622                         0.660
company                                0.291                         0.476
designation                            0.373                         0.571
DOC MEAN                               0.595                         0.678
DOC CORRECT                            0.032                         0.032
Saved /home/shaanal/Documents/EHS/Project_Resume_Parser/resume_parser_spacy/evaluat

## Summary of this run

In [4]:
import json
from pathlib import Path
r = lambda name: json.loads(Path("evaluation/results", name).read_text())
ner, sec, fields = r("ner_test.json"), r("sections.json"), r("fields_test.json")
print(ner["approach"])
print(f"Entity micro F1 (31 test resumes): exact {ner['micro']['exact']['f']:.3f} | overlap {ner['micro']['overlap']['f']:.3f} | {ner['ms_per_resume']:.0f} ms/resume")
for row in sec["results"]:
    print(f"Sections, {row['system']:<16} accuracy {row['accuracy']:.3f}  macro-F1 {row['macro_f1']:.3f}")
base, full = fields["results"]["baseline (rules + ontology)"], fields["results"]["full system"]
print(f"\n{'FIELD':<14}{'baseline':>10}{'full':>8}")
for name in full["fields"]:
    b, f = base["fields"][name]["score"], full["fields"][name]["score"]
    print(f"{name:<14}{b:>10.3f}{f:>8.3f}")
print(f"{'DOC MEAN':<14}{base['document_mean_score']:>10.3f}{full['document_mean_score']:>8.3f}")

A: spaCy CNN NER + TF-IDF/LogReg sections + char n-gram skill matching
Entity micro F1 (31 test resumes): exact 0.494 | overlap 0.593 | 73 ms/resume
Sections, rules only       accuracy 0.579  macro-F1 0.416
Sections, classifier only  accuracy 0.431  macro-F1 0.365
Sections, hybrid           accuracy 0.629  macro-F1 0.502

FIELD           baseline    full
full_name          0.857   0.893
email              1.000   1.000
location           0.567   0.567
skills             0.376   0.456
degree             0.603   0.756
institution        0.622   0.660
company            0.291   0.476
designation        0.373   0.571
DOC MEAN           0.595   0.678


## Parse the sample resumes and refresh the sample outputs

Batch output in `data/parsed/` uses the lean schema; add `--with-metadata` to keep confidence, evidence and the detected sections.

In [5]:
!python3 batch_parse.py

Components: {'approach': 'A: spaCy CNN NER + TF-IDF/LogReg sections + char n-gram skill matching', 'ner_model': 'models/resume_ner_spacy_vectors/model-best', 'section_model': 'models/section_classifier/section_classifier.joblib'}


[SUCCESS] 1.txt


[SUCCESS] 10.docx


[SUCCESS] 14.docx


[SUCCESS] 15.docx


...

[SUCCESS] CV_RashidSaid.pdf


[SUCCESS] CV_Robert Cooper.jpg


[SUCCESS] CV_ShihabAmri.pdf


[SUCCESS] CV_SoukeurAbdelhamid.jpg


[SUCCESS] CV_Yassine.pdf
Completed 33/33


In [6]:
!python3 -m inference.cli data/input/CV_GhandiAli.pdf --output sample_output.json > /dev/null
!python3 -m inference.cli data/input/CV_GhandiAli.pdf --with-metadata --output sample_output_with_metadata.json > /dev/null
!python3 schema.py sample_output.json sample_output_with_metadata.json

sample_output.json: OK
sample_output_with_metadata.json: OK


## Web demo and API

Run this in a terminal, not here, because the server keeps running: `uvicorn api.app:app --reload` (add `--port 8001` if the other project's server is on 8000).

Open http://127.0.0.1:8000, choose a resume and click **Parse**. The result appears as readable sections, with **Show JSON** / **Download JSON** for the raw output. The JSON API is documented at http://127.0.0.1:8000/docs (`POST /parse-resume`, optional `include_metadata=true`).